Настройка окружения

In [ ]:
import os

from dotenv import load_dotenv

# Импорт основных компонентов
from langchain_gigachat.chat_models import GigaChat
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

загружаем ключ из .env

In [ ]:
load_dotenv()

API_KEY = (os.getenv("GIGA_KEY") or "").strip()
giga_scope = (os.getenv("GIGACHAT_SCOPE") or "GIGACHAT_API_PERS").strip()

if not API_KEY or API_KEY == "ваш_ключ":
    raise ValueError("Укажите реальный GIGA_KEY в .env")

Убедись, что все работает

In [ ]:
# Настройка модели GigaChat с расширенными параметрами
llm = GigaChat(
    credentials=API_KEY,
    scope=giga_scope,
    model="GigaChat-Pro",  # можно указать конкретную версию, по умолчанию Lite
    verify_ssl_certs=False,
    temperature=0.1,  # настройка креативности
    max_tokens=1000,  # максимальная длина ответа
)

In [ ]:
# Проверка подключения
response = llm.invoke("Привет! Как дела?")
print(response.content)

## Задача: извлечение количества проживающих из заявок

Компания по аренде жилья получает тысячи заявок в виде неструктурированного текста.  
Нужно автоматически извлекать ключевую информацию — **количество проживающих**.

**Анализ сложностей:**
- вариативность формулировок количества людей;
- неявные указания («семья с двумя детьми» = 4 человека);
- отсутствие прямого указания количества в некоторых текстах;
- необходимость возвращения именно **целочисленного** результата.

### Базовое промптирование

In [ ]:
# Простой промпт для извлечения количества людей
basic_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
Проанализируй следующий текст заявки на аренду жилья и извлеки количество человек, которые будут проживать.
Текст заявки: {text}
Верни только число (целое число), соответствующее количеству проживающих.
Если количество не указано явно, постарайся определить его по контексту.
Количество человек:""",
)

# Создание цепочки
chain = basic_prompt | llm | StrOutputParser()

In [ ]:
# Индивидуальная работа: 15 заявок из rental_04.csv (вариант 04)
import csv

test_texts = {}
with open("rental_04.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter=";")
    for i, row in enumerate(reader, start=1):
        if i > 15:
            break
        test_texts[i] = row["text"].strip()

for idx, text in test_texts.items():
    result = chain.invoke({"text": text})
    print(f"#{idx}")
    print(f"Текст: {text}")
    print(f"Результат: {result}")
    print("---")

### Загрузка в DataFrame

`df` хранит все заявки из файла.
- столбец `text` содержит текст заявки;
- столбец `amount` хранит правильный ответ.

In [ ]:
import pandas as pd

# Загрузка данных из csv в датафрейм df
df = pd.read_csv("rental_04.csv", sep=";")

# Просмотр первых 5 строк
df.head()

### Обработка заявок в цикле и сохранение результатов

Проходим по столбцу `text`, отправляем каждую заявку в модель, 
сохраняем ответы в новый столбец `result` и записываем всю таблицу в CSV.

In [ ]:
results = []

for _, row in df.iterrows():
    text = row["text"]  # текст заявки
    try:
        result = chain.invoke({"text": text})
        results.append(result)
    except Exception as e:
        results.append(f"ERROR: {e}")

df["result"] = results

df.to_csv("rental_with_results.csv", index=False, encoding="utf-8-sig")

### Оценка точности модели

Сравниваем правильные ответы `amount` и предсказания модели `result`,
считаем количество ошибок и точность в процентах.

In [ ]:
print(df.dtypes)

result_num = pd.to_numeric(df["result"], errors="coerce")
amount_num = pd.to_numeric(df["amount"], errors="coerce")

correct_mask = result_num == amount_num

total = len(df)
correct = int(correct_mask.sum())
errors = int((~correct_mask).sum())
accuracy = correct / total if total else 0.0

print(f"Ошибок: {errors}")
print(f"Точность: {accuracy:.1%}")

## Дополнительное задание: структурированное извлечение полей

Ниже используем `ChatPromptTemplate` с сообщениями `System` и `Human` 
и возвращаем строго структурированный JSON по схеме.

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field


class RentalExtraction(BaseModel):
    count_adults: Optional[int] = Field(default=None, description="Количество взрослых")
    count_children: Optional[int] = Field(default=None, description="Количество детей")
    start_date: Optional[str] = Field(default=None, description="Дата заезда в формате DD.MM")
    nights: Optional[int] = Field(default=None, description="Количество ночей")
    price_per_day: Optional[int] = Field(default=None, description="Желаемая цена в сутки")
    remarks: Optional[str] = Field(default=None, description="Особые пожелания")


parser = PydanticOutputParser(pydantic_object=RentalExtraction)

In [ ]:
structured_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
Ты извлекаешь параметры заявки на аренду жилья.
Верни только JSON по схеме.

Правила:
1) count_adults — число взрослых.
2) count_children — число детей.
3) start_date — дата заезда в формате DD.MM.
   Если диапазон/плавающие даты, бери самую раннюю возможную дату.
4) nights — число ночей проживания.
   Если указано только число дней, используй его как nights.
   Если диапазон дат, вычисли разницу между датами в ночах.
   Если диапазон плавающий, возьми максимальное число ночей.
5) price_per_day — желаемая цена за сутки (если диапазон цен, взять максимум).
6) remarks — важные пожелания (море, бассейн, животные, локация и т.д.).
7) Если поля нет, возвращай null.
""",
        ),
        (
            "human",
            """
Текст заявки:
{text}

Формат ответа:
{format_instructions}
""",
        ),
    ]
)

structured_chain = structured_prompt | llm | parser

In [ ]:
# Берем 15 заявок варианта 04
work_df = pd.read_csv("rental_04.csv", sep=";").head(15).copy()

structured_results = []
for _, row in work_df.iterrows():
    text = row["text"]
    try:
        parsed = structured_chain.invoke(
            {
                "text": text,
                "format_instructions": parser.get_format_instructions(),
            }
        )
        structured_results.append(parsed.model_dump())
    except Exception as e:
        structured_results.append(
            {
                "count_adults": None,
                "count_children": None,
                "start_date": None,
                "nights": None,
                "price_per_day": None,
                "remarks": f"ERROR: {e}",
            }
        )

pred_df = pd.DataFrame(structured_results)
result_df = pd.concat([work_df.reset_index(drop=True), pred_df], axis=1)
result_df.head()

In [ ]:
# Ручная разметка для 15 заявок (эталон)
gold_data = [
    {"count_adults": 2, "count_children": 0, "start_date": "31.07", "nights": 10, "price_per_day": 1500},
    {"count_adults": 2, "count_children": 2, "start_date": "18.08", "nights": 6, "price_per_day": None},
    {"count_adults": 4, "count_children": 2, "start_date": "05.09", "nights": 12, "price_per_day": None},
    {"count_adults": 1, "count_children": 1, "start_date": "11.08", "nights": 11, "price_per_day": None},
    {"count_adults": 3, "count_children": 0, "start_date": "20.06", "nights": 8, "price_per_day": None},
    {"count_adults": 2, "count_children": 1, "start_date": "17.08", "nights": 10, "price_per_day": None},
    {"count_adults": 2, "count_children": 2, "start_date": "01.09", "nights": 9, "price_per_day": None},
    {"count_adults": 2, "count_children": 0, "start_date": "10.08", "nights": 8, "price_per_day": 2000},
    {"count_adults": 2, "count_children": 0, "start_date": "24.07", "nights": 8, "price_per_day": None},
    {"count_adults": 3, "count_children": 0, "start_date": "25.07", "nights": 10, "price_per_day": None},
    {"count_adults": 2, "count_children": 2, "start_date": "04.07", "nights": 20, "price_per_day": None},
    {"count_adults": 3, "count_children": 1, "start_date": "27.06", "nights": 8, "price_per_day": None},
    {"count_adults": 2, "count_children": 3, "start_date": "10.08", "nights": 13, "price_per_day": 3500},
    {"count_adults": 1, "count_children": 0, "start_date": "26.05", "nights": 19, "price_per_day": None},
    {"count_adults": 3, "count_children": 0, "start_date": "08.08", "nights": 12, "price_per_day": None},
]

gold_df = pd.DataFrame(gold_data)

# Метрики по полям 1-4
metrics_fields = ["count_adults", "count_children", "start_date", "nights"]

field_accuracies = {}
for col in metrics_fields:
    field_accuracies[col] = (result_df[col] == gold_df[col]).mean()

avg_accuracy = sum(field_accuracies.values()) / len(field_accuracies)

for col, score in field_accuracies.items():
    print(f"{col}: {score:.1%}")

print(f"Средняя точность по полям 1-4: {avg_accuracy:.1%}")

In [ ]:
# Сохраняем расширенный результат
result_df.to_csv("rental_with_structured_results.csv", index=False, encoding="utf-8-sig")
result_df[["text", "count_adults", "count_children", "start_date", "nights", "price_per_day", "remarks"]].head(15)